# Happy, sad and surprised faces from a sentence

This notebook extends the happy and sad notebook,

https://colab.research.google.com/github/eth-bmai-hs26/w4-cx-public/blob/main/happy-sad-diffusion/happy_sad_diffusion.ipynb

in two steps.

* A **third expression**, surprised, with the mouth wide open. The prompt becomes a **one-hot** of three numbers: $[1, 0, 0]$ asks for a happy face, $[0, 1, 0]$ for a sad one, $[0, 0, 1]$ for a surprised one. The forward process, the denoiser, the training and the generation are those of the first notebook.
* A **sentence as the prompt**. BERT, a transformer trained by Google, turns a sentence into $768$ numbers. One linear layer on top learns from $30$ short descriptions which expression a sentence asks for, and its answer, as a one-hot, becomes the prompt of the denoiser.

There is nothing to fill in: run the cells from top to bottom, then try your own sentences at the end.

In [ ]:
import torch
from torch import nn
import matplotlib.pyplot as plt

torch.manual_seed(0)   # seeded: the same noise and the same starting weights on every computer

## 1. The faces

The happy and sad faces of the first notebook, and a surprised one. Each face may again sit one pixel higher or lower and one pixel further left or right: $9$ faces per expression, $27$ in all. Each face comes with its one-hot.

In [ ]:
EYES = [(2, 3), (2, 5)]                                          # (row, column) of every ink pixel
HAPPY = [(5, 2), (6, 3), (6, 4), (6, 5), (5, 6)]                 # the mouth curves up
SAD = [(6, 2), (5, 3), (5, 4), (5, 5), (6, 6)]                   # the mouth curves down
SURPRISED = [(5, 3), (5, 4), (5, 5), (6, 3), (6, 5), (7, 3), (7, 4), (7, 5)]   # the mouth is wide open
EXPRESSIONS = ["happy", "sad", "surprised"]

faces, prompts = [], []
for one_hot, mouth in [([1, 0, 0], HAPPY), ([0, 1, 0], SAD), ([0, 0, 1], SURPRISED)]:
    for dy in [-1, 0, 1]:                                        # one pixel up, none, one pixel down
        for dx in [-1, 0, 1]:                                    # one pixel left, none, one pixel right
            face = torch.zeros(9, 9)
            for row, col in EYES + mouth:
                face[row + dy, col + dx] = 1.0
            faces.append(face.flatten())                         # the 81 pixels in one row
            prompts.append(one_hot)
faces = torch.stack(faces)                                       # 27 faces, 81 numbers each
prompts = torch.tensor(prompts, dtype=torch.float32)             # 27 one-hots, 3 numbers each
print(faces.shape, prompts.shape)

In [ ]:
def show(ax, picture, title=""):
    ax.imshow(picture.reshape(9, 9), cmap="Greys", vmin=0, vmax=1)
    ax.set(xticks=[], yticks=[], title=title)


fig, axes = plt.subplots(3, 9, figsize=(13, 5))
for ax, face, one_hot in zip(axes.flat, faces, prompts):
    show(ax, face, str(one_hot.int().tolist()))
plt.show()

## 2. The forward process and the denoiser

As in the first notebook, with one change: the network reads $81 + 1 + 3 = 85$ numbers, the noisy picture, the step $t$ (as $t/T$) and the one-hot.

In [ ]:
T = 20                                                           # the number of steps from a face to pure noise


def add_noise(face, noise, t):
    return (1 - t / T) * face + (t / T) * noise                 # the picture at step t


net = nn.Sequential(
    nn.Linear(81 + 1 + 3, 128), nn.ReLU(),                       # in: the noisy picture, the step and the one-hot
    nn.Linear(128, 128), nn.ReLU(),
    nn.Linear(128, 81),                                          # out: its guess of the clean face
)


def previous_picture(x, t, prompt):
    guess = net(torch.cat([x, t / T, prompt], dim=1))            # the network's guess of the clean face
    return x + (guess - x) / t                                   # one step from x_t towards it: the estimate of x_{t-1}

## 3. Training

The same loop and the same threshold as in the first notebook, now over the $27$ faces. It takes a few seconds longer.

In [ ]:
optimizer = torch.optim.Adam(net.parameters(), lr=0.001)

loss = 1.0                                                       # any start above the threshold
while loss > 0.012:
    pick = torch.randint(0, 27, (256,))                          # 256 faces picked at random
    face, prompt = faces[pick], prompts[pick]
    noise = torch.randn_like(face)                               # a noise picture for each face
    t = torch.randint(1, T + 1, (256, 1)).float()                # a step between 1 and T for each face
    x_t = add_noise(face, noise, t)                              # the noisy picture at step t
    x_previous = add_noise(face, noise, t - 1)                   # the true picture one step earlier
    loss = ((previous_picture(x_t, t, prompt) - x_previous) ** 2).sum(dim=1).mean()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f"\rloss: {loss.item():.4f}", end="")

## 4. Generating faces from a one-hot

The three steps of the slides, with a one-hot as the prompt. `torch.eye(3)` holds the three one-hots, one per row.

In [ ]:
def generate(one_hot, how_many=10):
    with torch.no_grad():
        x = torch.randn(how_many, 81)                            # 1) sample noise: x_T
        prompt = one_hot.repeat(how_many, 1)                     # the same one-hot for every face
        t = T
        while t > 0:                                             # 2) for t = T, T-1, ..., 1:
            step = torch.full((how_many, 1), float(t))
            x = previous_picture(x, step, prompt)                #    estimate x_{t-1} from x_t
            t = t - 1
    return x                                                     # 3) output x_0

In [ ]:
fig, axes = plt.subplots(3, 10, figsize=(15, 5))
for row, one_hot in zip(axes, torch.eye(3)):                     # the three one-hots, one per row
    for ax, picture in zip(row, generate(one_hot)):
        show(ax, picture, str(one_hot.int().tolist()))
plt.show()

## 5. BERT turns a sentence into 768 numbers

BERT is a transformer, like the ones of weekend 2, trained by Google on English Wikipedia and thousands of books. It cuts a sentence into tokens, words or pieces of words, puts a token of its own, `[CLS]`, in front and another, `[SEP]`, at the end. For every token it computes a vector of $768$ numbers, and the vector of `[CLS]` stands for the whole sentence. BERT is downloaded once, $440$ MB, and used as it is: nothing in it is trained here. Two messages come with the download, and both can be ignored: a warning that no `HF_TOKEN` is set (BERT is public, it needs no token), and a load report that lists seven weights as `UNEXPECTED` (they belong to the two tasks BERT was trained on, guessing hidden words and telling whether one sentence follows another, which this notebook does not need).

In [ ]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")   # cuts a sentence into tokens and numbers them
bert = AutoModel.from_pretrained("bert-base-uncased")            # BERT itself, 110 million weights


def embed(sentence):
    with torch.no_grad():                                        # BERT is used as it is, never trained here
        ids = tokenizer(sentence, return_tensors="pt").input_ids # the numbers of the tokens, [CLS] first
        vectors = bert(ids).last_hidden_state                    # a vector of 768 numbers for every token
    return vectors[0, 0]                                         # the vector of [CLS], for the whole sentence


ids = tokenizer("a flabbergasted woman", return_tensors="pt").input_ids
print(tokenizer.convert_ids_to_tokens(ids[0]))                   # the tokens BERT sees
print(embed("a flabbergasted woman").shape)

## 6. Which face does a sentence ask for?

Thirty short descriptions, ten for each expression, made up for this notebook. BERT turns each into its $768$ numbers, and one linear layer learns to give each sentence three scores, one per expression. The loss, cross-entropy, is small when the score of the right expression is well above the other two. It trains in under a second.

In [ ]:
HAPPY_TEXTS = ["a happy face", "a smiling face", "someone who is glad", "a cheerful person", "she is delighted",
               "he is laughing", "a big smile", "a pleased child", "they look very happy", "a grinning man"]
SAD_TEXTS = ["a sad face", "an unhappy face", "someone who is crying", "a gloomy person", "she is upset",
             "he is miserable", "tears running down a face", "a heartbroken child", "they look very sad",
             "a tearful woman"]
SURPRISED_TEXTS = ["a surprised face", "an astonished face", "someone who is amazed", "a shocked person",
                   "she is stunned", "he is startled", "a mouth wide open in surprise", "a speechless child",
                   "they look very surprised", "an open-mouthed man"]

embeddings, labels = [], []
for label, texts in [(0, HAPPY_TEXTS), (1, SAD_TEXTS), (2, SURPRISED_TEXTS)]:
    for text in texts:
        embeddings.append(embed(text))
        labels.append(label)
embeddings = torch.stack(embeddings)                             # 30 sentences, 768 numbers each
labels = torch.tensor(labels)                                    # 0 happy, 1 sad, 2 surprised
print(embeddings.shape, labels.shape)

In [ ]:
classifier = nn.Linear(768, 3)                                   # 768 numbers in, one score per expression out
optimizer = torch.optim.Adam(classifier.parameters(), lr=0.001)

loss = 1.0                                                       # any start above the threshold
while loss > 0.01:
    loss = nn.functional.cross_entropy(classifier(embeddings), labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f"\rloss: {loss.item():.4f}", end="")

## 7. From a sentence to a face

The expression with the best score becomes the one-hot, and the one-hot goes to `generate`.

In [ ]:
def one_hot_from(sentence):
    with torch.no_grad():
        scores = classifier(embed(sentence))                     # one score for each expression
    one_hot = torch.zeros(3)
    one_hot[scores.argmax()] = 1.0                               # 1 for the expression with the best score
    return one_hot


print(one_hot_from("a joyful face"))

None of the sentences below is among the thirty: the first three use words for a feeling that the classifier never saw, the last three name no feeling at all.

In [ ]:
sentences = ["a blissful baby", "a sorrowful face", "a flabbergasted woman",
             "I just won the lottery", "my dog died yesterday", "I can't believe my eyes"]
fig, axes = plt.subplots(len(sentences), 8, figsize=(12, 10))
for row, sentence in zip(axes, sentences):
    one_hot = one_hot_from(sentence)
    for ax, picture in zip(row, generate(one_hot, how_many=8)):
        show(ax, picture)
    row[0].set_title(f'"{sentence}": {EXPRESSIONS[one_hot.argmax()]}', loc="left")
plt.show()

## 8. Try it

* Write your own sentence in the cell below, a description or a situation, and run it.
* Write a negation, "a face that is not sad". Which face comes out, and why might that be?
* Add sentences of your own to the three lists of part 6, run that cell and the classifier again, and try the sentences that went wrong.

**What BERT brings.** The thirty sentences taught only the last layer which of BERT's directions means which face. Everything else BERT learned from its millions of sentences, which is why it also understands words it never saw here.

**What Stable Diffusion does differently.** It reads its prompt with the text half of CLIP, a transformer like BERT, and gives all of its vectors to the UNet, with no classifier in between, so it can draw what the sentence describes. The classifier here squeezes every sentence into one of three answers, so this model can only ever draw three faces.

In [ ]:
sentence = "a face full of surprise"                             # write your own sentence here
one_hot = one_hot_from(sentence)
print(EXPRESSIONS[one_hot.argmax()], one_hot.tolist())
fig, axes = plt.subplots(1, 10, figsize=(15, 1.9))
for ax, picture in zip(axes, generate(one_hot)):
    show(ax, picture)
plt.show()